# Q-Learning: From Zero to Hero

Welcome to Classical Reinforcement Learning. Unlike Supervised Learning, which relies on a static dataset of inputs and labels, Reinforcement Learning requires an agent to learn by interacting with an environment, receiving feedback in the form of positive or negative **rewards**.

### The Mathematical Core: The Bellman Equation
The agent updates its Q-Table after every single step using the Q-Learning update rule (derived from the Bellman Equation):

$$Q(s, a) \leftarrow Q(s, a) + \alpha [R + \gamma \max_{a'} Q(s', a') - Q(s, a)]$$

Where:
* $Q(s, a)$: The current value of taking action $a$ in state $s$.
* $\alpha$ (Alpha): The **Learning Rate** (how much we accept the new information vs old information).
* $R$: The immediate reward received after taking action $a$.
* $\gamma$ (Gamma): The **Discount Factor** (how much we care about future rewards vs immediate rewards).
* $\max_{a'} Q(s', a')$: The maximum expected future reward we can get from the *next* state $s'$.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing `gymnasium` and `numpy`.
2. **Environment Analysis:** Inspecting the FrozenLake grid world.
3. **Hyperparameter Initialization:** Setting up the Q-Table and math variables.
4. **The Training Loop:** Balancing Exploration vs. Exploitation ($\epsilon$-greedy).
5. **Evaluation:** Testing the trained, fully greedy agent.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, you must install: !pip install gymnasium numpy

import gymnasium as gym
import numpy as np
import random

print("Libraries imported successfully. Ready for Q-Learning.")

## Step 1: Initializing the Environment (FrozenLake)

We will use the classic **FrozenLake-v1** environment. 
Imagine a 4x4 grid of ice. 
* **S**: Start point, safe.
* **F**: Frozen surface, safe.
* **H**: Hole, fall to your doom (Reward: 0, Game Over).
* **G**: Goal, where the frisbee is (Reward: 1, Game Over).

By default, FrozenLake is "slippery". If you try to move Right, the ice might cause you to slip and move Up or Down instead. To make the underlying math of our tutorial easier to trace analytically, we will turn `is_slippery=False`, making it a deterministic environment.

In [ ]:
# Cell 4: Environment Analysis

# Create the environment. map_name="4x4" is the standard grid.
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)

# Reset environment to start
state, info = env.reset(seed=42)

print("--- Environment Analysis ---")
print(f"Observation Space (States): {env.observation_space.n}")
# There are 16 states (a 4x4 grid, numbered 0 to 15)

print(f"Action Space: {env.action_space.n}")
# There are 4 actions: 0=Left, 1=Down, 2=Right, 3=Up

# Let's see what taking a random action looks like
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)

print(f"\nTaking random action {action} from state 0:")
print(f"Landed in State: {next_state} | Reward: {reward} | Game Over: {terminated}")

## Step 2: The Q-Table and Hyperparameters

We need to create the Q-Table. Because we have 16 states and 4 actions, our Q-Table will be a 2D NumPy array of shape $(16, 4)$. We initialize all values to 0.

Next, we define our analytical parameters:
* **Epsilon ($\epsilon$):** We will use an $\epsilon$-greedy policy. Epsilon is the probability that the agent takes a completely random action (Exploration) instead of looking at the Q-Table (Exploitation). We will start $\epsilon$ high (1.0) and slowly decay it as the agent learns.

In [ ]:
# Cell 6: Initialization

# Initialize the Q-Table with zeros
action_space_size = env.action_space.n
state_space_size = env.observation_space.n
q_table = np.zeros((state_space_size, action_space_size))

print(f"Q-Table Shape initialized: {q_table.shape}")

# Algorithm Hyperparameters
num_episodes = 10000          # Total training episodes
max_steps_per_episode = 100   # Max steps before we force-quit an episode

learning_rate = 0.8           # Alpha
discount_rate = 0.95          # Gamma

# Exploration parameters
exploration_rate = 1.0        # Initial Epsilon
max_exploration_rate = 1.0    # Maximum Epsilon
min_exploration_rate = 0.01   # Minimum Epsilon
exploration_decay_rate = 0.001# How fast epsilon decays per episode

## Step 3: The Training Loop (Q-Learning Algorithm)

This is the core of the AI. For 10,000 episodes, we will let the agent play the game. 

For every single step:
1. Generate a random number. If it is less than $\epsilon$, pick a random action. Otherwise, pick the action with the highest Q-value for the current state.
2. Take the action in the environment.
3. Observe the new state and the reward.
4. **Apply the Bellman Update Rule** to update the old state's Q-value.
5. Decay $\epsilon$ so the agent explores less and exploits more over time.

In [ ]:
# Cell 8: Executing the Q-Learning Loop

print("Starting Training...")
rewards_all_episodes = []

for episode in range(num_episodes):
    # Reset the environment for a new episode
    state, info = env.reset()
    done = False
    rewards_current_episode = 0
    
    for step in range(max_steps_per_episode): 
        # 1. Exploration vs Exploitation trade-off
        exploration_rate_threshold = random.uniform(0, 1)
        
        if exploration_rate_threshold > exploration_rate:
            # EXPLOIT: Choose the action with the highest Q-value for this state
            action = np.argmax(q_table[state,:]) 
        else:
            # EXPLORE: Choose a random action
            action = env.action_space.sample()
            
        # 2. Take the action
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # 3. The Bellman Equation Update
        # Q(s,a) = Q(s,a) + alpha * [R + gamma * max(Q(s',a')) - Q(s,a)]
        q_table[state, action] = q_table[state, action] + learning_rate * \
            (reward + discount_rate * np.max(q_table[new_state, :]) - q_table[state, action])
        
        state = new_state
        rewards_current_episode += reward
        
        if done: 
            break
            
    # 4. Decay Epsilon exponentially after every episode
    exploration_rate = min_exploration_rate + \
        (max_exploration_rate - min_exploration_rate) * np.exp(-exploration_decay_rate*episode)
    
    rewards_all_episodes.append(rewards_current_episode)

print("Training Complete!")

## Step 4: Analytical Evaluation

Has the agent actually learned the environment? 

First, let's look at the mathematical artifact it generated: the final Q-Table. 
Then, we will run the agent for 100 test episodes with exploration turned completely off ($\epsilon = 0$). If it learned properly, it should navigate the ice and reach the goal 100% of the time without ever falling into a hole.

In [ ]:
# Cell 10: Evaluating the Trained Agent

# 1. Inspect the resulting Q-Table
print("--- Final Q-Table (Subset) ---")
print("State 0 (Start):", np.round(q_table[0, :], 4))
print("State 14 (Next to Goal):", np.round(q_table[14, :], 4))

# 2. Test the agent mathematically
test_episodes = 100
success_count = 0

for episode in range(test_episodes):
    state, info = env.reset()
    done = False
    
    for step in range(max_steps_per_episode):
        # ALWAYS EXPLOIT: We are testing the final learned policy
        action = np.argmax(q_table[state,:])
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        if done:
            if reward == 1:
                success_count += 1
            break
        state = new_state

print("\n--- Evaluation Results ---")
print(f"Agent successfully reached the goal in {success_count} out of {test_episodes} test episodes.")
success_rate = (success_count/test_episodes) * 100
print(f"Success Rate: {success_rate:.1f}%")

if success_rate == 100.0:
    print("\nAnalytical Conclusion: The Q-Learning algorithm perfectly mapped the environment and converged on the optimal mathematical policy!")
    
env.close()

## Conclusion

You have successfully implemented Q-Learning from scratch!

**Key Analytical Takeaways:**
1. **The Q-Table:** It is simply a matrix mapping States to Actions.
2. **The Bellman Equation:** This recursive formula allows the agent to pass information *backwards*. When it finally reaches the goal and gets a +1 reward, the Bellman equation mathematically propagates a fraction of that reward back to the state that preceded it, creating a breadcrumb trail of high Q-values leading to the goal.
3. **Exploration ($\epsilon$):** Without high initial exploration, the agent would just repeat the same safe, wrong moves forever. Randomness is a strict mathematical requirement for discovering optimal solutions.

While tabular Q-Learning is perfect for small grids, it fails when an environment has millions of states (like pixels on a screen). To solve that, we replace the `numpy` Q-Table matrix with a Neural Network, birthing **Deep Q-Networks (DQN)**!